# CS164 Session 2 — Taylor Series and Numerical Approximation

**Source:** reconstructed from the user-provided Session PDF screenshot, not an original `.ipynb` export. The visible student responses are retained as source responses; any Python plots are translations of the visible SageMath exercises, clearly identified below.


## Overview
The workbook uses Taylor expansions to approximate function values and derivatives numerically, and to help establish whether critical points are optima. For a sufficiently smooth univariate function, the order-$n$ Taylor polynomial about $x_0$ is
$$T_n(x;x_0)=\sum_{k=0}^n\frac{f^{(k)}(x_0)}{k!}(x-x_0)^k.$$


## Question 1 — When can a Taylor series be used?
**Submitted response:** the function must be infinitely differentiable at the expansion point; $f(x)=|x|$ at $x_0=0$ is a counterexample because it is not differentiable there.

**Technical qualification:** infinite differentiability is necessary to form all Taylor coefficients, but does not alone guarantee that the infinite series equals the function nearby.


## Question 2 — First-, second-, and third-order approximations to $\sin x$
The submitted work examines $x_0=\pi$ and $x_0=1$. At $\pi$, the linear and quadratic approximations coincide, $T_1=T_2=-(x-\pi)$; the cubic is $T_3=-(x-\pi)+(x-\pi)^3/6$. At $x_0=1$, derivatives yield
$$T_1=\sin1+\cos1(x-1),$$
$$T_2=T_1-\tfrac12\sin1(x-1)^2,$$
$$T_3=T_2-\tfrac16\cos1(x-1)^3.$$
The plots below translate the original two SageMath cells to Python.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def sin_taylor(x, x0, order):
    return sum(math.sin(x0+k*np.pi/2)/math.factorial(k)*(x-x0)**k
               for k in range(order+1))

def plot_sin_taylor(x0):
    x = np.linspace(x0-2, x0+2, 400)
    plt.figure(figsize=(8, 5))
    plt.plot(x, np.sin(x), label='sin(x)', linewidth=2.5)
    for k in [1,2,3]:
        plt.plot(x, sin_taylor(x, x0, k), linestyle='--', label=f'Taylor order {k}')
    plt.title(f'Expansion around x0={x0:.4f}')
    plt.xlabel('x'); plt.ylabel('f(x)'); plt.legend(); plt.grid(True); plt.show()

plot_sin_taylor(np.pi)
plot_sin_taylor(1.0)


## Multivariate Taylor polynomials
For a point $\mathbf x_0$, the linear approximation is
$$L(\mathbf x)=f(\mathbf x_0)+\nabla f(\mathbf x_0)^T(\mathbf x-\mathbf x_0),$$
and the quadratic approximation adds
$$\frac12(\mathbf x-\mathbf x_0)^T H_f(\mathbf x_0)(\mathbf x-\mathbf x_0).$$
The Hessian $H_f$ is the matrix of second partial derivatives.


## Question 3 — $f(x,y)=x^3+y$ at $(0,0)$
**Submitted result:** $\nabla f(0,0)=(0,1)$ and $H_f(0,0)=0$, hence both the linear and quadratic approximations are $L(x,y)=Q(x,y)=y$. The first distinction from the original function appears in the cubic term.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

def plot_three_surfaces(funcs, xlim, ylim, title):
    axis_x = np.linspace(*xlim, 55)
    axis_y = np.linspace(*ylim, 55)
    X, Y = np.meshgrid(axis_x, axis_y)
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(projection='3d')
    for fun, name, color in funcs:
        ax.plot_surface(X,Y,fun(X,Y),alpha=.4,color=color,linewidth=0)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=color,label=name,alpha=.5)
                       for _,name,color in funcs])
    ax.set(xlabel='x',ylabel='y',zlabel='z',title=title)
    plt.show()

plot_three_surfaces([
    (lambda x,y:x**3+y,'Original','C0'),
    (lambda x,y:y,'Linear','C1'),
    (lambda x,y:y,'Quadratic','C2')],(-1,1),(-1,1),'x³+y near (0,0)')


## Question 4 — $f(x,y)=\sin x+\cos y$ at two expansion points
The screenshot's two points are $(0,0)$ and $(\pi/2,\pi/2)$.

At $(0,0)$: $L=1+x$ and $Q=1+x-\frac12 y^2$.

At $(\pi/2,\pi/2)$: $L=1-(y-\pi/2)$ and $Q=1-(y-\pi/2)-\frac12(x-\pi/2)^2$.

The first-order Taylor surfaces are tangent planes; the second-order approximations also capture local curvature. Higher-order terms generally improve *local* approximation, though not necessarily far from the center.


In [ ]:
def trig_original(x,y):
    return np.sin(x)+np.cos(y)

def trig_linear(x,y,x0,y0):
    return np.sin(x0)+np.cos(y0)+np.cos(x0)*(x-x0)-np.sin(y0)*(y-y0)

def trig_quadratic(x,y,x0,y0):
    return trig_linear(x,y,x0,y0)-.5*np.sin(x0)*(x-x0)**2-.5*np.cos(y0)*(y-y0)**2

for x0,y0 in ((0.,0.),(np.pi/2,np.pi/2)):
    plot_three_surfaces([
        (trig_original,'Original','C0'),
        (lambda x,y,x0=x0,y0=y0:trig_linear(x,y,x0,y0),'Linear','C1'),
        (lambda x,y,x0=x0,y0=y0:trig_quadratic(x,y,x0,y0),'Quadratic','C2')],
        (x0-2,x0+2),(y0-2,y0+2),f'Taylor surfaces at ({x0:.3g},{y0:.3g})')


**Porting note:** The original workbook plotted the above surfaces in SageMath; the runnable cells here are equivalent NumPy/Matplotlib renderings. Only source-visible questions and results have been transcribed.
